# Phase 1: Supervised Fine-Tuning (SFT) - Core Foundations

This notebook performs the first phase of fine-tuning, training **Qwen 2.5-7B-Instruct** on Layers 1, 2, and 3 (General instruction, Coding & Mathematics, and CS/AI/ML core academic subjects) using **Unsloth** and **QLoRA** on a T4 GPU.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install Unsloth and training dependencies
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install datasets wandb

In [ ]:
# Setup logging and hubs
import wandb
from huggingface_hub import login
from google.colab import userdata

# Login to HF (save your HF_TOKEN in Colab Secrets)
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except Exception:
    print("HuggingFace token not found in Secrets. Please login manually if needed.")

# Login to WandB (save your WANDB_API_KEY in Colab Secrets)
try:
    wandb_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_key)
except Exception:
    print("WandB key not found in Secrets. Running without wandb log.")

In [ ]:
# Load Model and Tokenizer using Unsloth
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
dtype = torch.float16 # T4 GPU does not support bfloat16!
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# Configure LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 128,               # Large rank to hold substantial curriculum information
    lora_alpha = 256,      # Alpha = 2 * r
    target_modules = [     # Target all linear layers for maximum performance
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Enable memory saving
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
# Load preprocessed dataset from Google Drive
from datasets import load_dataset

dataset = load_dataset(
    "json", 
    data_files="/content/drive/MyDrive/btech-ai-tutor/data/phase1_train.jsonl", 
    split="train"
)
print(f"Loaded dataset with {len(dataset)} samples.")

In [ ]:
# Setup Training Arguments and Trainer
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 16, # Effective batch size = 64
    warmup_ratio = 0.03,
    num_train_epochs = 2,
    learning_rate = 2e-4,
    fp16 = True,
    bf16 = False, # Set to False since T4 does not support BF16
    logging_steps = 10,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 42,
    output_dir = "/content/drive/MyDrive/btech-ai-tutor/checkpoints/phase1",
    save_strategy = "steps",
    save_steps = 500,
    save_total_limit = 2,
    report_to = "wandb" if wandb.run is not None else "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True, # Pack multiple short conversations into a single sequence
    args = training_args,
)

In [ ]:
# Start training
# If resuming from a checkpoint on Google Drive, set resume_from_checkpoint=True
trainer.train(resume_from_checkpoint=False)

In [ ]:
# Save LoRA adapter to Google Drive
adapter_path = "/content/drive/MyDrive/btech-ai-tutor/adapters/phase1_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Saved adapter to {adapter_path}")

In [ ]:
# Merge and Save full model
# Note: Merging saves the model in FP16 to Google Drive for the next stage
merged_path = "/content/drive/MyDrive/btech-ai-tutor/models/phase1_merged"
model.save_pretrained_merged(merged_path, tokenizer, save_method = "merged_16bit")
print(f"Saved merged model to {merged_path}")